In [1]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from glob import glob
import os
import numpy as np
#import tensorflow
import torch
import pandas as pd
import torchaudio

from tqdm.notebook import tqdm

In [2]:
#! pip install git+https://github.com/openai/whisper.git
import whisper

In [5]:
mouseFLAC  = glob("vctk/gen/flac/wav48_silence_trimmed/*/*.flac")
mouseTXT = [m_fl.replace("_mic2.flac", ".txt").replace("vctk/gen/flac/wav48_silence_trimmed", "vctk/stock/txt") for m_fl in mouseFLAC]

mouseS = []
for i in mouseTXT:
    with open(i, 'r') as f:
        mouseS.append(f.read())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: {}".format(device))
torch.set_default_device(device)

device: cuda


In [6]:
import datasets
from datasets import Dataset, Audio

df = pd.DataFrame({"audio": mouseFLAC, "sentence": mouseS})
dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))


In [7]:
from huggingface_hub import notebook_login

notebook_login()

In [8]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="English", task="transcribe")



from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="English", task="transcribe")



In [9]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

do_lower_case = False
do_remove_punctuation = False

normalizer = BasicTextNormalizer()



#print(dataset["audio"][0][0])


#model = whisper.load_model("medium.en")




In [10]:
from transformers import WhisperProcessor

def prepare_dataset(batch):
    # load and (possibly) resample audio data to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array 
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    # compute input length of audio sample in seconds
    batch["input_length"] = len(audio["array"]) / audio["sampling_rate"]
    
    # optional pre-processing steps
    transcription = batch["sentence"]
    if do_lower_case:
        transcription = transcription.lower()
    if do_remove_punctuation:
        transcription = normalizer(transcription).strip()
    
    # encode target text to label ids
    batch["labels"] = processor.tokenizer(transcription).input_ids
    return batch



In [15]:


dataset = dataset.map(prepare_dataset, num_proc=2)



Map (num_proc=2):   0%|          | 0/3835 [00:00<?, ? examples/s]

2023-11-30 02:03:05.397012: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-30 02:03:05.401531: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-30 02:03:06.504867: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-11-30 02:03:06.504988: W tensorflow/compiler/xla/stream_executor/platfor

TimeoutError: 

In [5]:
n = 5

for i in range(n):
    PATH = mouseFLAC[i]
    with open(mouseTXT[i], 'r') as f:
        transcript = f.read().strip()
        
        result = model.transcribe(PATH)
        print("{} \t\t: {}".format( i, PATH))
        print("real \t\t: {}".format(transcript))
        print("transcription \t: {}".format(result["text"]))
        print()

0 		: vctk/gen/flac/wav48_silence_trimmed/p240/p240_257_mic2.flac
real 		: Her presence was almost everywhere.
transcription 	: 

1 		: vctk/gen/flac/wav48_silence_trimmed/p240/p240_240_mic2.flac
real 		: But it's a subtle process.
transcription 	: 

2 		: vctk/gen/flac/wav48_silence_trimmed/p240/p240_109_mic2.flac
real 		: It had been played at festivals.
transcription 	:  FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART FART F

3 		: vctk/gen/flac/wav48_silence_trimmed/p240/p240_216_mic2.flac